In [2]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda:0" 

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-0.5B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
prompt = "详细介绍下你自己。"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
# 将消息应用到标准模板，使结构一致
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(f"{input_text}") 

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
详细介绍下你自己。<|im_end|>
<|im_start|>assistant



In [4]:
model_inputs = tokenizer([input_text], return_tensors="pt").to(device)
print(f"{model_inputs}")

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 113511,  16872, 107828,   1773,
         151645,    198, 151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [5]:
response_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512,
)
print(f"resonse_ids: {response_ids}")

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


resonse_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 113511,  16872, 107828,   1773,
         151645,    198, 151644,  77091,    198, 100622,  15469, 110498,   3837,
         108763,  99605, 100798,  57191, 104934, 101904,   3837, 102114, 101099,
          47606,   1773,  35946, 110266,  74220,   3837,  99250,  70500, 102688,
         100364,  20002,  45912,  27369,   5373, 102104,  86119,  33108,  99553,
         100143,   1773, 109944, 102104, 100646,  31905, 103936,   3837, 100630,
         100022, 100032,   5373,  99891, 100032,   5373,  99348, 100032,   5373,
          99424, 107537,  49567,  90395, 100136, 100006,  99553, 105470,  85329,
          33108, 102011,  36407, 100364,  87026, 105344, 115167, 107124,   1773,
         106870, 110117,  86119,  57191,  85106, 100364,  37945, 102422, 106525,
           3837, 105351, 110121, 113445, 100143,  33108, 106185,   1773, 151645]],
       device

In [6]:
response_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, response_ids)]

print(f"response_ids: {response_ids}")
response_text = tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]
print(f"{response_text}")

response_ids: [tensor([100622,  15469, 110498,   3837, 108763,  99605, 100798,  57191, 104934,
        101904,   3837, 102114, 101099,  47606,   1773,  35946, 110266,  74220,
          3837,  99250,  70500, 102688, 100364,  20002,  45912,  27369,   5373,
        102104,  86119,  33108,  99553, 100143,   1773, 109944, 102104, 100646,
         31905, 103936,   3837, 100630, 100022, 100032,   5373,  99891, 100032,
          5373,  99348, 100032,   5373,  99424, 107537,  49567,  90395, 100136,
        100006,  99553, 105470,  85329,  33108, 102011,  36407, 100364,  87026,
        105344, 115167, 107124,   1773, 106870, 110117,  86119,  57191,  85106,
        100364,  37945, 102422, 106525,   3837, 105351, 110121, 113445, 100143,
         33108, 106185,   1773, 151645], device='cuda:0')]
作为AI助手，我没有个人经历或情感体验，也没有身体存在。我只是一个程序，被设计用来帮助用户获取信息、回答问题和提供支持。我可以回答各种类型的问题，包括历史知识、科学知识、文化知识、生活常识等，并且能够提供相关的资源和工具来帮助您更好地理解和解决问题。如果您有任何问题或需要帮助，请随时告诉我，我会尽力为您提供支持和解答。


In [7]:
generate_ids = model_inputs.input_ids

In [8]:
def generate_next_token(model, generate_ids, temperature=0.7, debug=False):
    print("input_ids:", generate_ids) if debug else None
    logits = model.forward(generate_ids).logits
    print("logits:", logits) if debug else None

    if temperature > 0:
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        print("probs:", len(probs[0]), probs) if debug else None
        next_token = torch.multinomial(probs[-1], num_samples=1)
    else:
        next_token = torch.argmax(logits[:, -1], dim=-1)
    print("next_id:", next_token, ", token:", tokenizer.decode(next_token))if debug else None
    return next_token.reshape(-1, 1)

In [9]:
next_token = generate_next_token(model, generate_ids, debug=True)
generate_ids = torch.cat((generate_ids, next_token), dim=1)
tokenizer.batch_decode(generate_ids[:, len(model_inputs.input_ids[0]):], skip_special_token=True)[0]

input_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 113511,  16872, 107828,   1773,
         151645,    198, 151644,  77091,    198]], device='cuda:0')
logits: tensor([[[ 2.2188,  3.0312,  3.2656,  ..., -3.4844, -3.4844, -3.4844],
         [ 0.6328,  1.9062,  3.4062,  ..., -4.5000, -4.4688, -4.5000],
         [ 8.1875,  4.6562, 10.8750,  ..., -3.3906, -3.3906, -3.3906],
         ...,
         [ 5.3125,  2.0156,  5.3438,  ..., -2.0469, -2.0469, -2.0469],
         [ 9.6875,  5.4062,  4.2500,  ..., -3.9531, -3.9375, -3.9375],
         [ 5.9375, 10.1250,  5.4062,  ..., -3.4219, -3.4219, -3.4219]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)
probs: 151936 tensor([[3.2887e-09, 1.2517e-06, 1.5061e-09,  ..., 5.1070e-15, 5.1070e-15,
         5.1070e-15]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SoftmaxBackward0>)
next_id: tensor([100622], device='cuda:0') , t

'作为'

In [13]:
import time
max_new_tokens = 128
end_token_id = 151645
generate_ids = model_inputs.input_ids

for _ in range(max_new_tokens):
    next_token = generate_next_token(model, generate_ids, debug=False)
    generate_ids = torch.cat((generate_ids, next_token), dim=1)
    if next_token.item() == end_token_id:
        break
    print(tokenizer.decode(next_token[0], skip_special_tokens=False), end='')

作为AI助手，我是一个由大量的计算和存储资源组成的程序，可以处理大量数据和任务。我使用自然语言处理技术来理解用户的问题并提供答案。我还可以帮助用户生成文本、撰写文章、提��建议和回答问题。

In [17]:
params = model.state_dict()
params
params['model.layers.0.self_attn.q_proj.weight']

tensor([[-0.0024, -0.0156,  0.0229,  ..., -0.0079, -0.0128,  0.0023],
        [ 0.0200,  0.0100, -0.0208,  ...,  0.0234,  0.0090, -0.0013],
        [-0.0151, -0.0035,  0.0219,  ..., -0.0092, -0.0283,  0.0043],
        ...,
        [-0.0415, -0.0522, -0.0233,  ...,  0.0427, -0.0374,  0.0063],
        [-0.0398, -0.0284, -0.0083,  ...,  0.0378, -0.0408, -0.0204],
        [ 0.0491, -0.0129,  0.0015,  ..., -0.0439,  0.0197,  0.0352]],
       device='cuda:0', dtype=torch.bfloat16)